# Goal Behaviour: L2 Escape Model

The following models takes into account that, throughout analyses, *no_goal* seems to behave entirely differently to the other two Gaol Type conditions (*goal_frequent* and *goal_non_frequent*). To do so, it splits the data accordingly and looks into goals on the one hand and no_goals in the other separately. 

As a reminder, Consequent Reversibility was not included in the final escape/ resolution models it is partially encoded by Goal_Type and produced unstable estimates when entered together.

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Overall Subgroup
This subset of data explores observations when _Focus_ = I and _Focus_= They.

1. Create escape_L2 for goals and no_goal
2. Counts and Proportions for goals and no_goal
3. Logistic Models
    - E1: Agency (goal_frequent vs. goal_non_frequent)
    - E2: Goal Type (goal_frequent vs. goal_non_frequent)
    - E3: Additive Model (goal_frequent vs. goal_non_frequent)
    - E4: Interactive Model (goal_frequent vs. goal_non_frequent)
    - E5: Agency (no_goal)
4. Model Comparison
5. Summary Table

Read Data

In [2]:
subgroup_theoretical = pd.read_csv("../../data/processed/subgroup_theoretical.csv")

1. Create escape_2

Binary Outcome: include responses that are not "L2_other": those who managed to escape L2_other by choosing either a *correct* response or an *L1_transfer* response.

In [3]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

Split data into goals vs. no_goals

In [4]:
goals = subgroup_theoretical[subgroup_theoretical["Goal_Type"]!= "no_goal"].copy()

In [5]:
no_goal = subgroup_theoretical[subgroup_theoretical["Goal_Type"]== "no_goal"].copy()

2. Counts and Proportions for goals and no_goal.

General escape_L2 reponses count:

In [6]:
escape_L2_counts = pd.crosstab(
    subgroup_theoretical["escape_L2"],
    subgroup_theoretical["Response_Full"])

escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,38,0,0
1,96,0,61,3


Escape_L2 responses count for goals:

In [7]:
goal_escape_L2_counts = pd.crosstab(
    goals["escape_L2"],
    goals["Response_Full"])

goal_escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,26,0,0
1,62,0,43,1


Escape_L2 responses count for no_goals:

In [8]:
no_goal_escape_L2_counts = pd.crosstab(
    no_goal["escape_L2"],
    no_goal["Response_Full"])

no_goal_escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,12,0,0
1,34,0,18,2


General escape_L2 responses proportions:

In [9]:
escape_L2_props = subgroup_theoretical["escape_L2"].value_counts(normalize = True)
escape_L2_props

escape_L2
1    0.808081
0    0.191919
Name: proportion, dtype: float64

From 198 observations, 80.8% of responses managed to escape L2_other options. 

Escape_L2 responses proportions for goals:

In [10]:
goals_escape_L2_props = goals["escape_L2"].value_counts(normalize = True)
goals_escape_L2_props

escape_L2
1    0.80303
0    0.19697
Name: proportion, dtype: float64

From 132 observations in the goal group, 80.3% of responses managed to escape L2_other options. 

Escape_L2 responses proportions for no_goals:

In [11]:
no_goal_escape_L2_props = no_goal["escape_L2"].value_counts(normalize = True)
no_goal_escape_L2_props

escape_L2
1    0.818182
0    0.181818
Name: proportion, dtype: float64

From 66 observations in the no_goal group, 81.8% of responses managed to escape L2_other options. 

Counts and Proportions by Condition:

In [12]:
goals_counts_condition = pd.crosstab(
    [goals["Goal_Type"], goals["Agent"]],
    goals["escape_L2"],
    margins = True)

goals_counts_condition

escape_L2                 0    1  All
Goal_Type         Agent              
goal_frequent     0      14   28   42
                  1       2   22   24
goal_non_frequent 0       9   33   42
                  1       1   23   24
All                      26  106  132

In [13]:
no_goal_counts_condition = pd.crosstab(
    no_goal["Agent"],
    no_goal["escape_L2"],
    margins = True)

no_goal_counts_condition

escape_L2,0,1,All
Agent,,,
0,8,34,42
1,4,20,24
All,12,54,66


In [14]:
goal_props_condition = pd.crosstab(
    [goals["Goal_Type"], goals["Agent"]],
    goals["escape_L2"],
    normalize= "index")

goal_props_condition

escape_L2                       0         1
Goal_Type         Agent                    
goal_frequent     0      0.333333  0.666667
                  1      0.083333  0.916667
goal_non_frequent 0      0.214286  0.785714
                  1      0.041667  0.958333

Within goals, Agency seems to assist in the process of escaping the L2_other options.

As previously mentioned, the condition with the strongest L1 transfer (goal_non_frequent, agent=1) exhibits the highest L2 escape rate (95.8%). This suggests that failure does not occur at the L2 suppression stage, but at the subsequent resolution stage.

In [15]:
no_goal_props_condition = pd.crosstab(
    no_goal["Agent"],
    no_goal["escape_L2"],
    normalize= "index")

no_goal_props_condition

escape_L2,0,1
Agent,,
0,0.190476,0.809524
1,0.166667,0.833333


Within no_goals, however, Agency doesn't seem to play the same part in the process of escaping the L2_other options compared to the goal group.

Agency appears to help participants escape L2. 

As previously mentioned, if there is a goal representation available, agency seems to help participants commit by leaving the L2 space; however, if there is no goal representation, agency seems to have much less to work with.

3. Logistic Models

Import Libraries

In [16]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2


Sanity checks:

In [17]:
goals["escape_L2"].value_counts()

escape_L2
1    106
0     26
Name: count, dtype: int64

In [18]:
no_goal["escape_L2"].value_counts()

escape_L2
1    54
0    12
Name: count, dtype: int64

**Model E1**

escape_L2 ~ Agent

In [19]:
E1 = smf.logit(
    "escape_L2 ~ Agent",
    data=goals
    ).fit()

print(E1.summary())

Optimization terminated successfully.
         Current function value: 0.458568
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                  132
Model:                          Logit   Df Residuals:                      130
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.07579
Time:                        10:56:14   Log-Likelihood:                -60.531
converged:                       True   LL-Null:                       -65.495
Covariance Type:            nonrobust   LLR p-value:                  0.001628
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9754      0.245      3.986      0.000       0.496       1.455
Agent          1.7327      0.

The model with Agency as the only predictor shows that there is a a main effect of Agency on escape_L2 responses.

In [20]:
# GEE
gee_E1 = smf.gee("escape_L2 ~ C(Agent)", 
                  groups="Participant_ID", 
                  data= goals, 
                  family=sm.families.Binomial())
results_1 = gee_E1.fit()
print(results_1.summary())

                               GEE Regression Results                              
Dep. Variable:                   escape_L2   No. Observations:                  132
Model:                                 GEE   No. clusters:                       33
Method:                        Generalized   Min. cluster size:                   4
                      Estimating Equations   Max. cluster size:                   4
Family:                           Binomial   Mean cluster size:                 4.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 07 Jul 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         10:56:14
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.9754      0.280      3.488      0.000       0.427       1.52

**Model E2**

escape_L2 ~ Goal_Type

In [21]:
E2 = smf.logit(
    "escape_L2 ~ Goal_Type",
    data= goals
    ).fit()

print(E2.summary())

Optimization terminated successfully.
         Current function value: 0.489593
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                  132
Model:                          Logit   Df Residuals:                      130
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.01326
Time:                        10:56:14   Log-Likelihood:                -64.626
converged:                       True   LL-Null:                       -65.495
Covariance Type:            nonrobust   LLR p-value:                    0.1875
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          1.1394      0.287      3.967     

The model with Goal Type as the only predictor shows that Goal Type on its own doesn not have an effect on L2_responses.

In [22]:
E3 = smf.logit(
    "escape_L2 ~ Agent + Goal_Type",
    data= goals
    ).fit()

print(E3.summary())

Optimization terminated successfully.
         Current function value: 0.451528
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                  132
Model:                          Logit   Df Residuals:                      129
Method:                           MLE   Df Model:                            2
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.08998
Time:                        10:56:14   Log-Likelihood:                -59.602
converged:                       True   LL-Null:                       -65.495
Covariance Type:            nonrobust   LLR p-value:                  0.002758
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.6854      0.317      2.162     

In [23]:
# GEE
gee_E3 = smf.gee("escape_L2 ~ C(Agent) + C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= goals, 
                  family=sm.families.Binomial())
results_1 = gee_E3.fit()
print(results_1.summary())

                               GEE Regression Results                              
Dep. Variable:                   escape_L2   No. Observations:                  132
Model:                                 GEE   No. clusters:                       33
Method:                        Generalized   Min. cluster size:                   4
                      Estimating Equations   Max. cluster size:                   4
Family:                           Binomial   Mean cluster size:                 4.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 07 Jul 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         10:56:14
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                             0.

The effect of Agency and Goal Type is also significant; however, the additive model doesn't seem to be better at predicting escape_L2 responses than the model with Agency alone. 

**Model E4**

escape_L2 ~ Agent * Goal_Type

In [24]:
E4 = smf.logit(
    "escape_L2 ~ Agent * Goal_Type",
    data=goals
    ).fit()

print(E4.summary())

Optimization terminated successfully.
         Current function value: 0.451492
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                  132
Model:                          Logit   Df Residuals:                      128
Method:                           MLE   Df Model:                            3
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.09005
Time:                        10:56:14   Log-Likelihood:                -59.597
converged:                       True   LL-Null:                       -65.495
Covariance Type:            nonrobust   LLR p-value:                  0.008117
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                                0.6931      0.3

While the overall interactivve model's performance also shows significance, it doesn't perform better than the model with agency alone.

**Model E5 (no_goal only)**

escape_L2 ~ Agent 

In [25]:
E5 = smf.logit(
    "escape_L2 ~ Agent",
    data=no_goal
    ).fit()

print(E5.summary())

Optimization terminated successfully.
         Current function value: 0.473694
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   66
Model:                          Logit   Df Residuals:                       64
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:               0.0009392
Time:                        10:56:14   Log-Likelihood:                -31.264
converged:                       True   LL-Null:                       -31.293
Covariance Type:            nonrobust   LLR p-value:                    0.8084
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.4469      0.393      3.682      0.000       0.677       2.217
Agent          0.1625      0.

The model confirms that Agency does not help participants at all in the process of escaping L2 responses when there isn't a goal structure. 

4. Model Comparisons

In [26]:
from scipy.stats import chi2

def lr_test(model_small, model_large):

    LR = 2 * (model_large.llf - model_small.llf)

    df = model_large.df_model - model_small.df_model

    p = chi2.sf(LR, df)

    return pd.Series({
        "LR": LR,
        "df": df,
        "p_value": p
    })

In [27]:
lr_test(E1, E3)

LR         1.858509
df         1.000000
p_value    0.172797
dtype: float64

In [28]:
lr_test(E1, E4)

LR         1.868002
df         2.000000
p_value    0.392978
dtype: float64

5. Summary Table

In [29]:
goal_behaviour = ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"]

model_name = ["E1", "E2", "E3", "E3", "E4", "E4", "E4", "E5"]

outcome = ["Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2"]

subgroup = ["Overall", "Overall", "Overall", "Overall", "Overall", "Overall", "Overall", "Overall"]

formula = ["escape_L2 ~ Agent", "escape_L2 ~ Goal_Type", "escape_L2 ~ Agent + Goal_Type", "escape_L2 ~ Agent + Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent"]

predictor= ["Agent", "goal_non_frequent", "goal_non_frequent", "Agent",  "goal_non_frequent", "Agent", "Agent:goal_non_frequent", "Agent"]

coef = [1.7327, 0.5833, 0.6242, 1.7527, 0.6061, 1.7047, 0.1315, 0.1625]

std_error = [0.645, 0.448, 0.463, 0.648, 0.499, 0.808, 1.356, 0.674]

coef_p = [0.007, 0.193, 0.178, 0.007, 0.224, 0.035, 0.923, 0.809]

LLR_p = [0.001628, 0.1875, 0.002758, 0.002758, 0.008117, 0.008117, 0.008117, 0.8084]

N= [132, 132, 132, 132, 132, 132, 132, 66]

Takeaway = []

In [30]:
gb_escape_summary_overall = pd. DataFrame({
    "Goal_Behaviour": goal_behaviour,
    "Model": model_name,
    "Outcome": outcome,
    "Subgroup": subgroup,
    "Formula": formula, 
    "Predictor": predictor,
    "Coef": coef,
    "Std_err": std_error,
    "Coef_p": coef_p,
    "Model_LLR_p": LLR_p,
    "N": N
})

gb_escape_summary_overall

,Goal_Behaviour,Model,Outcome,Subgroup,Formula,Predictor,Coef,Std_err,Coef_p,Model_LLR_p,N
0,Yes,E1,Escape_L2,Overall,escape_L2 ~ Agent,Agent,1.7327,0.645,0.007,0.001628,132
1,Yes,E2,Escape_L2,Overall,escape_L2 ~ Goal_Type,goal_non_frequent,0.5833,0.448,0.193,0.187500,132
2,Yes,E3,Escape_L2,Overall,escape_L2 ~ Agent + Goal_Type,goal_non_frequent,0.6242,0.463,0.178,0.002758,132
3,Yes,E3,Escape_L2,Overall,escape_L2 ~ Agent + Goal_Type,Agent,1.7527,0.648,0.007,0.002758,132
4,Yes,E4,Escape_L2,Overall,escape_L2 ~ Agent * Goal_Type,goal_non_frequent,0.6061,0.499,0.224,0.008117,132
5,Yes,E4,Escape_L2,Overall,escape_L2 ~ Agent * Goal_Type,Agent,1.7047,0.808,0.035,0.008117,132
6,Yes,E4,Escape_L2,Overall,escape_L2 ~ Agent * Goal_Type,Agent:goal_non_frequent,0.1315,1.356,0.923,0.008117,132
7,Yes,E5,Escape_L2,Overall,escape_L2 ~ Agent,Agent,0.1625,0.674,0.809,0.808400,66


## Actors Subgroup
This subset of data explores observations only when _Focus_ = I with the aim of finding out whether models' performance improve once the variability that is brought upon when _Focus_ = They is eliminated. 

1. Create escape_L2 for goals vs. no_goal.
2. Counts and Proportions for goals vs. no_goal
3. Logistic Models
    - E1_A: Agency (goal_frequent vs. goal_non_frequent)
    - E2_A: Goal Type (goal_frequent vs. goal_non_frequent)
    - E3_A: Additive Model (goal_frequent vs. goal_non_frequent)
    - E4_A: Interactive Model (goal_frequent vs. goal_non_frequent)
    - E5_A: Agency (no_goal)
4. Model Comparison
5. Summary Table

Read Data

In [31]:
subgroup_actors = subgroup_theoretical[subgroup_theoretical["Focus"] == "I"].copy()

1. Create escape_L2

Binary Outcome: within actors, include responses that are not *L2_other*; that is, those who managed to escape *L2_other* by choosing either a *correct* response or an *L1_transfer* response.

In [32]:
subgroup_actors["escape_L2"] = (
    subgroup_actors["Response_Full"] != "L2_other"
).astype(int)

In [33]:
goals_actors = subgroup_actors[subgroup_actors["Goal_Type"]!= "no_goal"].copy()

In [34]:
no_goal_actors = subgroup_actors[subgroup_actors["Goal_Type"]== "no_goal"].copy()

2. Counts and Proportions

In [35]:
escape_L2_counts = pd.crosstab(
    subgroup_actors["escape_L2"],
    subgroup_actors["Response_Full"])

escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,24,0,0
1,53,0,40,3


In [36]:
goals_escape_L2_counts = pd.crosstab(
    goals_actors["escape_L2"],
    goals_actors["Response_Full"])

goals_escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,19,0,0
1,31,0,29,1


In [37]:
no_goal_escape_L2_counts = pd.crosstab(
    no_goal_actors["escape_L2"],
    no_goal_actors["Response_Full"])

no_goal_escape_L2_counts

Response_Full,L1_transfer,L2_other,correct,missing_response
escape_L2,,,,
0,0,5,0,0
1,22,0,11,2


In [38]:
escape_L2_props = subgroup_actors["escape_L2"].value_counts(normalize = True)
escape_L2_props

escape_L2
1    0.8
0    0.2
Name: proportion, dtype: float64

From 120 actors' observations (from the original 198), almost 80% of responses managed to escape *L2_other* options just as with the overall subgroup.

In [39]:
goals_escape_L2_props = goals_actors["escape_L2"].value_counts(normalize = True)
goals_escape_L2_props

escape_L2
1    0.7625
0    0.2375
Name: proportion, dtype: float64

In [40]:
no_goal_escape_L2_props = no_goal_actors["escape_L2"].value_counts(normalize = True)
no_goal_escape_L2_props

escape_L2
1    0.875
0    0.125
Name: proportion, dtype: float64

Counts and Proportions by Condition:

In [41]:
counts_condition = pd.crosstab(
    [subgroup_actors["Goal_Type"], subgroup_actors["Agent"]],
    subgroup_actors["escape_L2"],
    margins = True)

counts_condition

escape_L2                 0   1  All
Goal_Type         Agent             
goal_frequent     0       8  16   24
                  1       2  14   16
goal_non_frequent 0       8  16   24
                  1       1  15   16
no_goal           0       3  21   24
                  1       2  14   16
All                      24  96  120

In [42]:
goals_counts_condition = pd.crosstab(
    [goals_actors["Goal_Type"], goals_actors["Agent"]],
    goals_actors["escape_L2"],
    margins = True)

goals_counts_condition

escape_L2                 0   1  All
Goal_Type         Agent             
goal_frequent     0       8  16   24
                  1       2  14   16
goal_non_frequent 0       8  16   24
                  1       1  15   16
All                      19  61   80

In [43]:
no_goal_counts_condition = pd.crosstab(
    no_goal_actors["Agent"],
    no_goal_actors["escape_L2"],
    margins = True)

no_goal_counts_condition

escape_L2,0,1,All
Agent,,,
0,3,21,24
1,2,14,16
All,5,35,40


In [44]:
props_condition = pd.crosstab(
    [subgroup_actors["Goal_Type"], subgroup_actors["Agent"]],
    subgroup_actors["escape_L2"],
    normalize= "index")

props_condition

escape_L2                       0         1
Goal_Type         Agent                    
goal_frequent     0      0.333333  0.666667
                  1      0.125000  0.875000
goal_non_frequent 0      0.333333  0.666667
                  1      0.062500  0.937500
no_goal           0      0.125000  0.875000
                  1      0.125000  0.875000

Just as with the Overall subgroup, within actors, Agency also appears to help participants escape L2. 

The condition with the strongest L1 transfer (goal_non_frequent, agent=1) keeps exhibiting the highest L2 escape rate (93.7%, slightly lower than in the overall subgroup (95.8%)). Still, this keeps hinting at  failure not occuring at the L2 suppression stage, but at the subsequent resolution stage.

Considering differences regarding goal representation availability, these proportions support that, if there is a goal representation available, agency seems to help participants commit by leaving the L2 space; however, if there is no goal representation, agency has nothing to work with by showing the exact same proportions. 

This further supports the idea that we could be dealing with an entire different behaviour when it comes to escaping l2 (and subsequent resolving) when having a goal representation available (*goal_frequent* and *goal_non_frequent*) vs. not having a goal representation available (*no_goal*) that should be explored later on. 

In [45]:
goals_props_condition = pd.crosstab(
    [goals_actors["Goal_Type"], goals_actors["Agent"]],
    goals_actors["escape_L2"],
    normalize= "index")

goals_props_condition

escape_L2                       0         1
Goal_Type         Agent                    
goal_frequent     0      0.333333  0.666667
                  1      0.125000  0.875000
goal_non_frequent 0      0.333333  0.666667
                  1      0.062500  0.937500

In [46]:
no_goal_props_condition = pd.crosstab(
    no_goal_actors["Agent"],
    no_goal_actors["escape_L2"],
    normalize= "index")

no_goal_props_condition

escape_L2,0,1
Agent,,
0,0.125,0.875
1,0.125,0.875


3. Logistic Models

In [47]:
subgroup_actors["escape_L2"].value_counts()

escape_L2
1    96
0    24
Name: count, dtype: int64

In [48]:
goals_actors["escape_L2"].value_counts()

escape_L2
1    61
0    19
Name: count, dtype: int64

In [49]:
no_goal_actors["escape_L2"].value_counts()

escape_L2
1    35
0     5
Name: count, dtype: int64

**E1_A Model** 

escape_L2 ~ Agent

In [50]:
E1_A = smf.logit(
    "escape_L2 ~ Agent",
    data=goals_actors
    ).fit()

print(E1_A.summary())

Optimization terminated successfully.
         Current function value: 0.506360
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   80
Model:                          Logit   Df Residuals:                       78
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.07629
Time:                        10:56:14   Log-Likelihood:                -40.509
converged:                       True   LL-Null:                       -43.854
Covariance Type:            nonrobust   LLR p-value:                  0.009688
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6931      0.306      2.264      0.024       0.093       1.293
Agent          1.5755      0.

**E2_A Model** 

escape_L2 ~  Goal_Type

In [51]:
E2_A = smf.logit(
    "escape_L2 ~ Goal_Type",
    data=goals_actors
    ).fit()

print(E2_A.summary())

Optimization terminated successfully.
         Current function value: 0.547749
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   80
Model:                          Logit   Df Residuals:                       78
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:               0.0007873
Time:                        10:56:14   Log-Likelihood:                -43.820
converged:                       True   LL-Null:                       -43.854
Covariance Type:            nonrobust   LLR p-value:                    0.7927
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          1.0986      0.365      3.009     

**E3_A Model**

escape_L2 ~  Agent + Goal_Type

In [52]:
E3_A = smf.logit(
    "escape_L2 ~ Agent + Goal_Type",
    data= goals_actors
    ).fit()

print(E3_A.summary())

Optimization terminated successfully.
         Current function value: 0.505893
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   80
Model:                          Logit   Df Residuals:                       77
Method:                           MLE   Df Model:                            2
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.07714
Time:                        10:56:14   Log-Likelihood:                -40.471
converged:                       True   LL-Null:                       -43.854
Covariance Type:            nonrobust   LLR p-value:                   0.03394
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.6193      0.406      1.524     

**E4_A Model**

escape_L2 ~ Agent * Goal_Type

In [53]:
E4_A = smf.logit(
    "escape_L2 ~ Agent * Goal_Type",
    data= goals_actors
    ).fit()

print(E4_A.summary())

Optimization terminated successfully.
         Current function value: 0.504021
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   80
Model:                          Logit   Df Residuals:                       76
Method:                           MLE   Df Model:                            3
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.08056
Time:                        10:56:14   Log-Likelihood:                -40.322
converged:                       True   LL-Null:                       -43.854
Covariance Type:            nonrobust   LLR p-value:                   0.06984
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                                0.6931      0.4

**E5_A Model (no_goals only)**

escape_L2 ~ Agent 

In [54]:
E5_A = smf.logit(
    "escape_L2 ~ Agent",
    data= no_goal_actors
    ).fit()

print(E5_A.summary())

Optimization terminated successfully.
         Current function value: 0.376770
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              escape_L2   No. Observations:                   40
Model:                          Logit   Df Residuals:                       38
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:               1.086e-10
Time:                        10:56:14   Log-Likelihood:                -15.071
converged:                       True   LL-Null:                       -15.071
Covariance Type:            nonrobust   LLR p-value:                     1.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.9459      0.617      3.153      0.002       0.736       3.156
Agent      -5.528e-16      0.

4. Model comparisons

In [55]:
lr_test(E1_A, E3_A)

LR         0.074737
df         1.000000
p_value    0.784560
dtype: float64

In [56]:
lr_test(E1_A, E4_A)

LR         0.374288
df         2.000000
p_value    0.829324
dtype: float64

5. Summary Table

gb_escape_summary_actors

In [57]:
goal_behaviour = ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"]

model_name = ["E1_A", "E2_A", "E3_A", "E3_A", "E4_A", "E4_A", "E4_A", "E5_A"]

outcome = ["Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2", "Escape_L2"]

subgroup = ["Actors", "Actors", "Actors", "Actors", "Actors", "Actors", "Actors", "Actors"]

formula = ["escape_L2 ~ Agent", "escape_L2 ~ Goal_Type", "escape_L2 ~ Agent + Goal_Type", "escape_L2 ~ Agent + Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent * Goal_Type", "escape_L2 ~ Agent"]

predictor= ["Agent", "goal_non_frequent", "goal_non_frequent", "Agent",  "goal_non_frequent", "Agent", "Agent:goal_non_frequent", "Agent"]

coef = [1.5755, 0.1382, 0.1495, 1.5769, 4.935e-16, 1.2528, 0.7621, -5.528e-16 ]

std_error = [0.679, 0.526, 0.547, 0.680, 0.612, 0.871, 1.419, 0.976] 

coef_p = [0.020, 0.793, 0.785, 0.020, 1.000, 0.150, 0.591, 1.000]

LLR_p = [0.009688, 0.7927, 0.03394, 0.03394, 0.06984, 0.06984, 0.06984, 1.000]

N= [80, 80, 80, 80, 80, 80, 80, 40]

Takeaway = []

In [58]:
gb_escape_summary_actors = pd.DataFrame({
    "Goal_Behaviour": goal_behaviour,
    "Model": model_name,
    "Outcome": outcome,
    "Subgroup": subgroup,
    "Formula": formula, 
    "Predictor": predictor,
    "Coef": coef,
    "Std_err": std_error,
    "Coef_p": coef_p,
    "Model_LLR_p": LLR_p,
    "N": N
})

gb_escape_summary_actors

,Goal_Behaviour,Model,Outcome,Subgroup,Formula,Predictor,Coef,Std_err,Coef_p,Model_LLR_p,N
0,Yes,E1_A,Escape_L2,Actors,escape_L2 ~ Agent,Agent,1.575500e+00,0.679,0.020,0.009688,80
1,Yes,E2_A,Escape_L2,Actors,escape_L2 ~ Goal_Type,goal_non_frequent,1.382000e-01,0.526,0.793,0.792700,80
2,Yes,E3_A,Escape_L2,Actors,escape_L2 ~ Agent + Goal_Type,goal_non_frequent,1.495000e-01,0.547,0.785,0.033940,80
3,Yes,E3_A,Escape_L2,Actors,escape_L2 ~ Agent + Goal_Type,Agent,1.576900e+00,0.680,0.020,0.033940,80
4,Yes,E4_A,Escape_L2,Actors,escape_L2 ~ Agent * Goal_Type,goal_non_frequent,4.935000e-16,0.612,1.000,0.069840,80
5,Yes,E4_A,Escape_L2,Actors,escape_L2 ~ Agent * Goal_Type,Agent,1.252800e+00,0.871,0.150,0.069840,80
6,Yes,E4_A,Escape_L2,Actors,escape_L2 ~ Agent * Goal_Type,Agent:goal_non_frequent,7.621000e-01,1.419,0.591,0.069840,80
7,Yes,E5_A,Escape_L2,Actors,escape_L2 ~ Agent,Agent,-5.528000e-16,0.976,1.000,1.000000,40
